# Week 8: Southern Africa Case Study - Go/No-Go Recommendations

> **Milestone:** Case study with decision-ready go/no-go zones
> **Deliverable:** Notebook 06 complete - regional heatmap with go/no-go recommendations for Southern Africa

*From PROJECT_BRIEF.md - Phase 2: Optimization (Weeks 5-8)*

In [ ]:
import numpy as np
import rasterio
import geopandas as gpd
from shapely.geometry import box, Point
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os
import json
import folium
from folium import plugins

wd = '/home/recursivex/my_projects/my_career/2026_roadmap_fully_funded_opportunities/hydrogen_storage_site_selection'
os.chdir(wd)

# Load normalized data
solar_norm = np.load('outputs/solar_norm.npy')
water_norm = np.load('outputs/water_norm.npy')
pop_norm = np.load('outputs/pop_norm.npy')

# Create ROI polygon (Southern Africa: Namibia, Botswana, western South Africa)
roi = box(12.0, -20.0, 22.0, -10.0)
roi_gdf = gpd.GeoDataFrame({'geometry': [roi]}, crs='EPSG:4326')

# Create mask based on overlapping valid values
mask = (solar_norm > 0) & (water_norm > 0) & (pop_norm > 0)
solar_vals = solar_norm[mask]
water_vals = water_norm[mask]
pop_vals = pop_norm[mask]

print(f'Valid pixels for analysis: {len(solar_vals)}')
print(f'Solar range: {solar_vals.min():.4f} - {solar_vals.max():.4f}')
print(f'Water range: {water_vals.min():.4f} - {water_vals.max():.4f}')
print(f'Pop range: {pop_vals.min():.4f} - {pop_vals.max():.4f}')

## Load Optimization Results

Load the multi-objective optimization results including weight configurations, scores, and robustness analysis.

In [ ]:
with open('outputs/optimization_results.json', 'r') as f:
    opt_results = json.load(f)

weights_array = np.array(opt_results['weights'])
scores = np.array(opt_results['scores'])
rep_configs = opt_results['representative_configs']

print(f'Loaded {len(weights_array)} weight configurations')
print(f'Scores shape: {scores.shape}')

## Equal-Weight Baseline Suitability Map

Compute the balanced (equal-weight) suitability score for every pixel in the ROI.

In [ ]:
# Equal weight baseline
equal_weight = np.array([1/3, 1/3, 1/3])
equal_scores_flat = solar_vals * equal_weight[0] + water_vals * equal_weight[1] + pop_vals * equal_weight[2]

# Reshape back to grid
equal_scores_grid = np.zeros_like(solar_norm)
equal_scores_grid[mask] = equal_scores_flat

print(f'Equal weight scores: min={equal_scores_flat.min():.4f}, max={equal_scores_flat.max():.4f}, mean={equal_scores_flat.mean():.4f}')

# Percentiles for threshold setting
percentiles = np.percentile(equal_scores_flat, [25, 50, 75, 90, 95])
for p, v in zip([25, 50, 75, 90, 95], percentiles):
    print(f'  {p}th percentile: {v:.4f}')

## Robustness Map

Identify pixels that are consistently suitable across different weight configurations.

In [ ]:
# Compute robustness: pixel in top 25% for >=70% of weight configurations
n_weights = len(weights_array)
pixel_top_count = np.zeros(len(solar_vals))
for i in range(n_weights):
    threshold = np.percentile(scores[i], 75)
    pixel_top_count += (scores[i] >= threshold).astype(int)

robust_threshold = int(n_weights * 0.7)
robust_pixels = pixel_top_count >= robust_threshold

# Reshape to grid
robust_grid = np.zeros_like(solar_norm)
robust_grid[mask] = robust_pixels.astype(float)

robust_count = np.sum(robust_pixels)
robust_ratio = robust_count / len(solar_vals) * 100

print(f'Robustly suitable pixels: {robust_count} out of {len(solar_vals)} ({robust_ratio:.1f}%)')

## Go/No-Go Decision Criteria

Define decision-ready thresholds based on the equal-weight baseline and robustness analysis:
- **GO (Highly Suitable):** Score >= 75th percentile AND robust
- **CONDITIONAL GO:** Score >= 75th percentile but NOT robust (weight-sensitive)
- **CONDITIONAL NO-GO:** Score >= 50th percentile but < 75th percentile
- **NO-GO:** Score < 50th percentile

In [ ]:
# Define thresholds
p75 = np.percentile(equal_scores_flat, 75)
p50 = np.percentile(equal_scores_flat, 50)

print(f'Decision thresholds:')
print(f'  75th percentile (GO threshold): {p75:.4f}')
print(f'  50th percentile (NO-GO threshold): {p50:.4f}')

# Classify each pixel
decision_grid = np.zeros_like(solar_norm)  # 0=No-Go, 1=Conditional No-Go, 2=Conditional Go, 3=Go
decision_flat = np.zeros(len(equal_scores_flat), dtype=int)

for i in range(len(equal_scores_flat)):
    score = equal_scores_flat[i]
    is_robust = robust_pixels[i]
    
    if score >= p75 and is_robust:
        decision_flat[i] = 3  # GO
    elif score >= p75 and not is_robust:
        decision_flat[i] = 2  # CONDITIONAL GO
    elif score >= p50:
        decision_flat[i] = 1  # CONDITIONAL NO-GO
    else:
        decision_flat[i] = 0  # NO-GO

decision_grid[mask] = decision_flat

# Count decisions
decision_counts = np.bincount(decision_flat)
labels = ['NO-GO', 'Conditional NO-GO', 'Conditional GO', 'GO']
for i, (label, count) in enumerate(zip(labels, decision_counts)):
    print(f'  {label}: {count} pixels ({count/len(decision_flat)*100:.1f}%)')

## Identify Candidate Regions (Connected Components)

Find contiguous GO and Conditional GO regions for site-specific analysis.

In [ ]:
from scipy import ndimage

# Find connected GO regions (values 2 and 3)
go_mask = (decision_grid >= 2).astype(int)
labeled, num_features = ndimage.label(go_mask)

print(f'Found {num_features} candidate regions (GO + Conditional GO)')

# Analyze each region
regions = []
for region_id in range(1, num_features + 1):
    region_mask = (labeled == region_id)
    region_pixels = np.sum(region_mask)
    
    if region_pixels < 10:  # Skip very small regions
        continue
    
    # Get coordinates of region center
    coords = np.argwhere(region_mask)
    center_y, center_x = coords.mean(axis=0)
    
    # Convert pixel indices to lat/lon
    # Assuming solar_norm has shape (height, width) and covers roi
    h, w = solar_norm.shape
    lon = roi.bounds[0] + (center_x / w) * (roi.bounds[2] - roi.bounds[0])
    lat = roi.bounds[3] - (center_y / h) * (roi.bounds[3] - roi.bounds[1])
    
    # Average scores in region
    region_scores = equal_scores_grid[region_mask]
    region_robust = robust_grid[region_mask]
    
    regions.append({
        'id': region_id,
        'size_pixels': int(region_pixels),
        'center_lon': float(lon),
        'center_lat': float(lat),
        'mean_score': float(region_scores.mean()),
        'robust_fraction': float(region_robust.mean()),
        'max_score': float(region_scores.max())
    })

# Sort by mean score
regions.sort(key=lambda x: x['mean_score'], reverse=True)

print(f'\nTop {min(5, len(regions))} candidate regions:')
for i, r in enumerate(regions[:5]):
    print(f'  Region {r["id"]}: ({r["center_lat"]:.3f}, {r["center_lon"]:.3f}), '
          f'size={r["size_pixels"]} px, '
          f'mean_score={r["mean_score"]:.3f}, '
          f'robust={r["robust_fraction"]:.2f}, '
          f'max={r["max_score"]:.3f}')

## Static Visualization: Decision Map

Create a publication-quality decision map showing go/no-go zones.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Equal-weight suitability
im1 = axes[0, 0].imshow(equal_scores_grid, cmap='RdYlGn', origin='upper', vmin=0, vmax=1)
axes[0, 0].set_title('Equal-Weight Suitability Score (1/3 each)', fontsize=12)
plt.colorbar(im1, ax=axes[0, 0], label='Suitability Score')

# 2. Robustness map
im2 = axes[0, 1].imshow(robust_grid, cmap='RdYlGn', origin='upper', vmin=0, vmax=1)
axes[0, 1].set_title('Robustness (Fraction of configs in top 25%)', fontsize=12)
plt.colorbar(im2, ax=axes[0, 1], label='Robustness')

# 3. Go/No-Go decision map
colors = ['red', 'orange', 'lightgreen', 'darkgreen']
from matplotlib.colors import ListedColormap
cmap_decision = ListedColormap(colors)
im3 = axes[1, 0].imshow(decision_grid, cmap=cmap_decision, origin='upper', vmin=-0.5, vmax=3.5)
axes[1, 0].set_title('Go/No-Go Decision Zones', fontsize=12)
cbar = plt.colorbar(im3, ax=axes[1, 0], ticks=[0, 1, 2, 3])
cbar.ax.set_yticklabels(['NO-GO', 'Cond. NO-GO', 'Cond. GO', 'GO'])

# 4. Top candidate regions
im4 = axes[1, 1].imshow(equal_scores_grid, cmap='RdYlGn', origin='upper', vmin=0, vmax=1, alpha=0.7)
for r in regions[:5]:
    # Convert lat/lon back to pixel coords for plotting
    h, w = solar_norm.shape
    px = int((r['center_lon'] - roi.bounds[0]) / (roi.bounds[2] - roi.bounds[0]) * w)
    py = int((roi.bounds[3] - r['center_lat']) / (roi.bounds[3] - roi.bounds[1]) * h)
    axes[1, 1].plot(px, py, 'ko', markersize=10, markerfacecolor='yellow', markeredgewidth=2)
    axes[1, 1].text(px+5, py+5, f'R{r["id"]}', fontsize=10, fontweight='bold', color='black')
axes[1, 1].set_title('Top 5 Candidate Regions for Tactical Analysis', fontsize=12)
plt.colorbar(im4, ax=axes[1, 1], label='Suitability Score')

plt.tight_layout()
plt.savefig('outputs/southern_africa_decision_map.png', dpi=150, bbox_inches='tight')
plt.close()

print('Decision map saved to outputs/southern_africa_decision_map.png')

## Interactive Folium Map

Create an interactive web map for stakeholder exploration.

In [ ]:
# Create folium map centered on Southern Africa
center_lat = (roi.bounds[1] + roi.bounds[3]) / 2
center_lon = (roi.bounds[0] + roi.bounds[2]) / 2

m = folium.Map(location=[center_lat, center_lon], zoom_start=5, tiles='CartoDB positron')

# Add suitability as heatmap
heat_data = []
h, w = solar_norm.shape
for i in range(0, h, 5):  # Subsample for performance
    for j in range(0, w, 5):
        if mask[i, j]:
            lat = roi.bounds[3] - (i / h) * (roi.bounds[3] - roi.bounds[1])
            lon = roi.bounds[0] + (j / w) * (roi.bounds[2] - roi.bounds[0])
            val = equal_scores_grid[i, j]
            if val > 0.3:  # Only show moderately suitable areas
                heat_data.append([lat, lon, float(val)])

plugins.HeatMap(heat_data, radius=10, blur=15, max_zoom=8).add_to(m)

# Add candidate region markers
for r in regions[:5]:
    folium.Marker(
        location=[r['center_lat'], r['center_lon']],
        popup=f"Region {r['id']}<br>Score: {r['mean_score']:.3f}<br>Robust: {r['robust_fraction']:.2f}<br>Size: {r['size_pixels']} px",
        icon=folium.Icon(color='green', icon='map-marker')
    ).add_to(m)

# Add ROI boundary
folium.GeoJson(
    roi_gdf,
    name='Study Area',
    style_function=lambda x: {'color': 'black', 'weight': 2, 'fillOpacity': 0}
).add_to(m)

folium.LayerControl().add_to(m)
m.save('outputs/southern_africa_interactive_map.html')

print('Interactive map saved to outputs/southern_africa_interactive_map.html')

## Representative Weight Set Analysis

Show how decisions change under different stakeholder preferences.

In [ ]:
# Compute decisions for each representative weight set
rep_names = ['solar_focus', 'water_focus', 'pop_focus', 'balanced']
rep_weights = [rep_configs[n]['weight'] for n in rep_names]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax_idx, (name, w) in enumerate(zip(rep_names, rep_weights)):
    scores_w = solar_vals * w[0] + water_vals * w[1] + pop_vals * w[2]
    scores_grid = np.zeros_like(solar_norm)
    scores_grid[mask] = scores_w
    
    im = axes[ax_idx].imshow(scores_grid, cmap='RdYlGn', origin='upper', vmin=0, vmax=1)
    axes[ax_idx].set_title(f'{name.replace("_", " ").title()}\nw={w}', fontsize=10)
    plt.colorbar(im, ax=axes[ax_idx])

plt.tight_layout()
plt.savefig('outputs/representative_weight_maps.png', dpi=150, bbox_inches='tight')
plt.close()

print('Representative weight maps saved!')

## Export Results for Tactical Planning Layer

Save candidate regions and decision map for integration with PINN-based tactical analysis (Phase 3).

In [ ]:
# Save case study results
case_study_results = {
    'roi_bounds': list(roi.bounds),
    'equal_weight': equal_weight.tolist(),
    'thresholds': {'p50': float(p50), 'p75': float(p75)},
    'decision_counts': {labels[i]: int(decision_counts[i]) for i in range(len(labels))},
    'robust_ratio': float(robust_ratio),
    'candidate_regions': regions[:5],
    'weight_sensitivities': opt_results.get('weight_sensitivities', []),
    'representative_weights': {name: rep_configs[name]['weight'] for name in rep_names}
}

with open('outputs/southern_africa_case_study.json', 'w') as f:
    json.dump(case_study_results, f, indent=2)

# Also save the grids for tactical layer
np.save('outputs/decision_grid.npy', decision_grid)
np.save('outputs/equal_scores_grid.npy', equal_scores_grid)
np.save('outputs/robust_grid.npy', robust_grid)

print('Case study results exported!')
print(json.dumps(case_study_results, indent=2))

## Week 8 Deliverables Summary

- **Decision map:** outputs/southern_africa_decision_map.png (static) and outputs/southern_africa_interactive_map.html (interactive)
- **Go/No-Go zones:** 4-class decision grid saved as outputs/decision_grid.npy
- **Candidate regions:** Top 5 regions identified for tactical site-specific analysis
- **Weight sensitivity maps:** outputs/representative_weight_maps.png
- **Export for Phase 3:** outputs/southern_africa_case_study.json with all metadata

---
*Next: Phase 3 Week 9 - PINN integration for H2-brine IFT prediction as safety penalty layer*